In [8]:
import requests

AGENTS = [
    "http://localhost:8000",
    "http://localhost:8001"
]

# Discover agents
def discover():
    data = []
    for url in AGENTS:
        info = requests.get(f"{url}/agent-info").json()
        tools = requests.get(f"{url}/tools").json()
        data.append({"info": info, "tools": tools})
    return data

# Route (A2A)
def route(query):
    if "interest" in query:
        return "finance-agent"
    return "math-agent"

# Invoke (MCP)
def invoke(agent_name, payload):
    for url in AGENTS:
        info = requests.get(f"{url}/agent-info").json()
        if info["name"] == agent_name:
            return requests.post(info["endpoint"], json=payload).json()

In [9]:
import re

def plan(query):
    query = query.lower()
    numbers = list(map(float, re.findall(r"\d+\.?\d*", query)))

    steps = []

    if "interest" in query:
        steps.append({
            "agent": "finance-agent",
            "payload": {
                "tool": "calculate_interest",
                "args": {
                    "principal": numbers[0],
                    "rate": numbers[1],
                    "time": numbers[2]
                }
            }
        })

    if "add" in query:
        steps.append({
            "agent": "math-agent",
            "payload": {
                "tool": "add",
                "args": {
                    "b": numbers[-1]  # last number = 500
                }
            }
        })

    return steps

In [10]:
def execute(query):
    steps = plan(query)

    print("\n📋 PLAN:", steps)

    result = None

    for i, step in enumerate(steps):

        payload = step["payload"]

        # inject previous result
        if payload["tool"] == "add":
            payload["args"]["a"] = result

        print(f"\n🚀 STEP {i+1}: {step['agent']} → {payload}")

        res = invoke(step["agent"], payload)

        result = res["result"]

        print("✅ RESULT:", result)

    return result

In [11]:
query = "calculate interest for 2000 at 10 for 1 year and then add 500"

print("\n🎯 FINAL:", execute(query))


📋 PLAN: [{'agent': 'finance-agent', 'payload': {'tool': 'calculate_interest', 'args': {'principal': 2000.0, 'rate': 10.0, 'time': 1.0}}}, {'agent': 'math-agent', 'payload': {'tool': 'add', 'args': {'b': 500.0}}}]

🚀 STEP 1: finance-agent → {'tool': 'calculate_interest', 'args': {'principal': 2000.0, 'rate': 10.0, 'time': 1.0}}
✅ RESULT: 200.0

🚀 STEP 2: math-agent → {'tool': 'add', 'args': {'b': 500.0, 'a': 200.0}}
✅ RESULT: 700.0

🎯 FINAL: 700.0
